# Long-tail / cold-start analysis for Steam recommendation

Цель ноутбука - проверить, где именно языковая семантика и QLoRA могут быть полезны:

```text
1. head / mid / tail games по популярности в train;
2. cold-ish games, то есть игры с малым числом train-взаимодействий;
3. users with short / medium / long history;
4. сравнение CatBoost, CatBoost + semantic features и QLoRA на доступной QLoRA test subset.
```

Ограничения честности:
- popularity считается только по train split;
- threshold выбирается только на validation;
- global Steam aggregates не используются;
- `target_hours`, target labels, `user_id`, `target_app_id` не используются как признаки моделей;
- `target_app_id` используется только для анализа групп, а не как feature.


In [ ]:
import os, json, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_score, recall_score, f1_score

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path("F:/Coursework")
print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
LABEL_COL = "label_strict"
ITEM_COL = "target_app_id"
ALPHA = 10.0
USE_GPU = False

DATA_DIR = PROJECT_ROOT / "data" / "final" / "tabular_temporal"
SEMANTIC_DIR = PROJECT_ROOT / "outputs" / "semantic"
SEMANTIC_ZIP = SEMANTIC_DIR / "steam_semantic_outputs.zip"
QLORA_PRED_PATH = PROJECT_ROOT / "outputs" / "qwen_qlora" / "checkpoint2500_eval" / "eval_outputs" / "qwen_checkpoint_test_predictions.csv"
OUT_DIR = PROJECT_ROOT / "outputs" / "analysis" / "long_tail_cold_start"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR:", OUT_DIR)
print("QLoRA predictions:", QLORA_PRED_PATH, "exists:", QLORA_PRED_PATH.exists())


## 1. Load tabular data


In [ ]:
train_raw = pd.read_parquet(DATA_DIR / "train_tabular.parquet")
val_raw = pd.read_parquet(DATA_DIR / "val_tabular.parquet")
test_raw = pd.read_parquet(DATA_DIR / "test_tabular.parquet")
print("tabular shapes:", train_raw.shape, val_raw.shape, test_raw.shape)
print("test label balance:", test_raw[LABEL_COL].value_counts().to_dict())


## 2. Train-only popularity features


In [ ]:
BASE_FEATURES = [
    "history_len", "history_positive_count", "history_negative_count", "history_positive_share",
    "history_total_hours", "history_mean_hours", "history_median_hours", "history_max_hours", "history_min_hours",
    "history_liked_mean_hours", "history_disliked_mean_hours", "target_token_count", "liked_token_count",
    "disliked_token_count", "target_liked_overlap_count", "target_disliked_overlap_count", "target_liked_jaccard",
    "target_disliked_jaccard", "target_jaccard_diff", "target_description_len", "target_title_len",
    "price", "required_age", "dlc_count", "release_year"
]
POP_FEATURES = ["train_item_popularity_score", "train_item_log_count"]

def build_item_stats(df, global_rate):
    stats = df.groupby(ITEM_COL)[LABEL_COL].agg(["sum", "count"]).reset_index()
    stats["train_item_popularity_score"] = (stats["sum"] + ALPHA * global_rate) / (stats["count"] + ALPHA)
    stats["train_item_log_count"] = np.log1p(stats["count"])
    return stats[[ITEM_COL, "train_item_popularity_score", "train_item_log_count"]]

def add_full_train_popularity(df, train):
    global_rate = float(train[LABEL_COL].mean())
    stats = build_item_stats(train, global_rate)
    result = df.merge(stats, on=ITEM_COL, how="left")
    result["train_item_popularity_score"] = result["train_item_popularity_score"].fillna(global_rate)
    result["train_item_log_count"] = result["train_item_log_count"].fillna(0.0)
    return result

def add_oof_train_popularity(train):
    result = train.copy()
    result["train_item_popularity_score"] = np.nan
    result["train_item_log_count"] = np.nan
    y = train[LABEL_COL].astype(int).values
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    for fit_idx, hold_idx in skf.split(train, y):
        fit_part = train.iloc[fit_idx]
        hold_part = train.iloc[hold_idx]
        global_rate = float(fit_part[LABEL_COL].mean())
        stats = build_item_stats(fit_part, global_rate)
        encoded = hold_part[[ITEM_COL]].merge(stats, on=ITEM_COL, how="left")
        result.iloc[hold_idx, result.columns.get_loc("train_item_popularity_score")] = encoded["train_item_popularity_score"].fillna(global_rate).values
        result.iloc[hold_idx, result.columns.get_loc("train_item_log_count")] = encoded["train_item_log_count"].fillna(0.0).values
    return result

train = add_oof_train_popularity(train_raw)
val = add_full_train_popularity(val_raw, train_raw)
test = add_full_train_popularity(test_raw, train_raw)
print("prepared:", train.shape, val.shape, test.shape)


## 3. Load and merge semantic features


In [ ]:
SEM_COLS = [
    "semantic_target_liked_cosine", "semantic_target_disliked_cosine", "semantic_similarity_gap",
    "semantic_max_liked_cosine", "semantic_max_disliked_cosine", "semantic_top3_liked_mean_cosine",
    "semantic_top3_disliked_mean_cosine"
]
expected_semantic_files = ["train_semantic_features.parquet", "val_semantic_features.parquet", "test_semantic_features.parquet"]
if SEMANTIC_ZIP.exists():
    missing = [name for name in expected_semantic_files if not list(SEMANTIC_DIR.rglob(name))]
    if missing:
        with zipfile.ZipFile(SEMANTIC_ZIP, "r") as z:
            z.extractall(SEMANTIC_DIR)
        print("Semantic zip extracted.")

def find_semantic_file(name):
    hits = list(SEMANTIC_DIR.rglob(name))
    if not hits:
        raise FileNotFoundError(f"Required semantic file was not found: {name}")
    return hits[0]

def merge_semantic(tab, sem):
    sem_cols = [c for c in SEM_COLS if c in sem.columns]
    if "sample_id" in tab.columns and "sample_id" in sem.columns:
        return tab.merge(sem[["sample_id"] + sem_cols], on="sample_id", how="left")
    if len(tab) != len(sem):
        raise ValueError("Semantic and tabular datasets have different number of rows; row-wise merge is not safe.")
    return pd.concat([tab.reset_index(drop=True), sem[sem_cols].reset_index(drop=True)], axis=1)

train_semantic = merge_semantic(train, pd.read_parquet(find_semantic_file("train_semantic_features.parquet")))
val_semantic = merge_semantic(val, pd.read_parquet(find_semantic_file("val_semantic_features.parquet")))
test_semantic = merge_semantic(test, pd.read_parquet(find_semantic_file("test_semantic_features.parquet")))
semantic_features = [c for c in SEM_COLS if c in train_semantic.columns]
print("semantic features:", semantic_features)
print(test_semantic[semantic_features].isna().mean())


## 4. Feature validation


In [ ]:
FORBIDDEN_EXACT = {
    "label_strict", "label_hybrid", "label_hours_relative", "is_recommended", "output", "target_hours",
    "target_hours_aux", "user_id", "target_app_id", "app_id", "review_id", "positive_ratio", "user_reviews",
    "positive", "negative", "recommendations", "peak_ccu", "average_playtime_forever", "median_playtime_forever",
    "pct_pos_total", "num_reviews_total", "pct_pos_recent", "num_reviews_recent"
}
FORBIDDEN_PATTERNS = ["label", "output", "target_hours", "review_id", "hours_relative", "hours_reference", "low_hours_positive", "high_hours_negative"]

def validate_features(df, features, name):
    missing = [c for c in features if c not in df.columns]
    if missing:
        raise ValueError(f"{name} has missing columns: {missing}")
    bad = []
    for c in features:
        cl = c.lower()
        if c in FORBIDDEN_EXACT or any(p in cl for p in FORBIDDEN_PATTERNS) or not pd.api.types.is_numeric_dtype(df[c]):
            bad.append(c)
    if bad:
        raise ValueError(f"{name} contains forbidden or non-numeric features: {bad}")

FEATURES_BASE = BASE_FEATURES + POP_FEATURES
FEATURES_SEM = FEATURES_BASE + semantic_features
validate_features(train, FEATURES_BASE, "FEATURES_BASE")
validate_features(train_semantic, FEATURES_SEM, "FEATURES_SEM")
print("features base:", len(FEATURES_BASE))
print("features semantic:", len(FEATURES_SEM))


## 5. Train CatBoost models


In [ ]:
def calc_metrics(y_true, score, threshold):
    pred = score >= threshold
    return {
        "roc_auc": float(roc_auc_score(y_true, score)),
        "pr_auc": float(average_precision_score(y_true, score)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "threshold": float(threshold),
    }

def choose_threshold(y_true, score):
    thresholds = np.linspace(0.0, 1.0, 1001)
    best_threshold, best_f1 = 0.5, -1.0
    for threshold in thresholds:
        cur_f1 = f1_score(y_true, score >= threshold, zero_division=0)
        if cur_f1 > best_f1:
            best_f1 = cur_f1
            best_threshold = threshold
    return float(best_threshold)

def train_eval_catboost(method, train_df, val_df, test_df, features):
    X_train, X_val, X_test = train_df[features], val_df[features], test_df[features]
    y_train = train_df[LABEL_COL].astype(int).values
    y_val = val_df[LABEL_COL].astype(int).values
    y_test = test_df[LABEL_COL].astype(int).values
    params = {
        "iterations": 1000, "learning_rate": 0.04, "depth": 6, "loss_function": "Logloss", "eval_metric": "AUC",
        "random_seed": SEED, "verbose": 100, "allow_writing_files": False, "early_stopping_rounds": 80,
    }
    if USE_GPU:
        params["task_type"] = "GPU"
        params["devices"] = "0"
    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
    val_score = model.predict_proba(X_val)[:, 1]
    test_score = model.predict_proba(X_test)[:, 1]
    threshold = choose_threshold(y_val, val_score)
    row = {"method": method, **{f"test_{k}": v for k, v in calc_metrics(y_test, test_score, threshold).items()}, "val_roc_auc": float(roc_auc_score(y_val, val_score)), "n_features": len(features), "best_iteration": model.get_best_iteration()}
    pred_test = pd.DataFrame({
        "sample_id": test_df["sample_id"].values if "sample_id" in test_df.columns else np.arange(len(test_df)),
        ITEM_COL: test_df[ITEM_COL].values,
        "y_true": y_test,
        "score": test_score,
        "pred": (test_score >= threshold).astype(int),
    })
    model.save_model(OUT_DIR / f"{method}.cbm")
    pred_test.to_csv(OUT_DIR / f"{method}_test_predictions.csv", index=False)
    pd.DataFrame({"feature": features, "importance": model.get_feature_importance()}).sort_values("importance", ascending=False).to_csv(OUT_DIR / f"{method}_feature_importance.csv", index=False)
    return row, pred_test

base_row, base_test_pred = train_eval_catboost("catboost_popularity", train, val, test, FEATURES_BASE)
sem_row, sem_test_pred = train_eval_catboost("catboost_popularity_semantic", train_semantic, val_semantic, test_semantic, FEATURES_SEM)
overall_metrics = pd.DataFrame([base_row, sem_row])
overall_metrics.to_csv(OUT_DIR / "overall_metrics.csv", index=False)
overall_metrics


## 6. Group labels


In [ ]:
item_counts = train_raw.groupby(ITEM_COL).size().rename("target_train_count").reset_index()
item_positive = train_raw.groupby(ITEM_COL)[LABEL_COL].mean().rename("target_train_positive_rate").reset_index()
item_stats = item_counts.merge(item_positive, on=ITEM_COL, how="left")

def add_group_columns(df):
    result = df.copy().merge(item_stats, on=ITEM_COL, how="left")
    result["target_train_count"] = result["target_train_count"].fillna(0).astype(int)
    result["target_train_positive_rate"] = result["target_train_positive_rate"].fillna(float(train_raw[LABEL_COL].mean()))
    result["item_count_bucket"] = pd.cut(
        result["target_train_count"], bins=[-1, 0, 2, 5, 20, 100, np.inf],
        labels=["cold_0", "very_rare_1_2", "rare_3_5", "tail_6_20", "mid_21_100", "head_100_plus"]
    ).astype(str)
    result["item_popularity_quantile"] = pd.qcut(
        result["target_train_count"].rank(method="first"), q=4,
        labels=["Q1_tail", "Q2_low_mid", "Q3_high_mid", "Q4_head"]
    ).astype(str)
    result["history_len_bucket"] = pd.cut(
        result["history_len"], bins=[-1, 2, 5, 10, 20, np.inf],
        labels=["hist_0_2", "hist_3_5", "hist_6_10", "hist_11_20", "hist_21_plus"]
    ).astype(str)
    return result

test_groups = add_group_columns(test_raw)
group_cols = ["item_count_bucket", "item_popularity_quantile", "history_len_bucket"]
for c in group_cols:
    print(c)
    print(test_groups[c].value_counts(dropna=False))


## 7. Group metrics


In [ ]:
def calc_group_metrics(y_true, score, pred):
    y_true, score, pred = np.asarray(y_true).astype(int), np.asarray(score), np.asarray(pred).astype(int)
    return {
        "n": int(len(y_true)),
        "positive_rate": float(np.mean(y_true)) if len(y_true) else np.nan,
        "accuracy": float(accuracy_score(y_true, pred)) if len(y_true) else np.nan,
        "precision": float(precision_score(y_true, pred, zero_division=0)) if len(y_true) else np.nan,
        "recall": float(recall_score(y_true, pred, zero_division=0)) if len(y_true) else np.nan,
        "f1": float(f1_score(y_true, pred, zero_division=0)) if len(y_true) else np.nan,
        "pr_auc": float(average_precision_score(y_true, score)) if len(np.unique(y_true)) > 1 else np.nan,
        "roc_auc": float(roc_auc_score(y_true, score)) if len(np.unique(y_true)) > 1 else np.nan,
    }

def group_metrics(pred_df, group_df, method, group_col):
    if "sample_id" in pred_df.columns and "sample_id" in group_df.columns:
        df = pred_df.merge(group_df[["sample_id", group_col]], on="sample_id", how="left")
    else:
        df = pred_df.copy(); df[group_col] = group_df[group_col].values
    rows = []
    for value, part in df.groupby(group_col, dropna=False):
        rows.append({"method": method, "group_type": group_col, "group": str(value), **calc_group_metrics(part["y_true"], part["score"], part["pred"])})
    return rows

rows = []
for group_col in group_cols:
    rows += group_metrics(base_test_pred, test_groups, "catboost_popularity", group_col)
    rows += group_metrics(sem_test_pred, test_groups, "catboost_popularity_semantic", group_col)
catboost_group_metrics = pd.DataFrame(rows)
catboost_group_metrics.to_csv(OUT_DIR / "catboost_group_metrics.csv", index=False)
catboost_group_metrics


## 8. QLoRA group metrics and fair subset comparison


In [ ]:
qlora_group_metrics = pd.DataFrame()
fair_subset_metrics = pd.DataFrame()
if QLORA_PRED_PATH.exists():
    qlora_pred = pd.read_csv(QLORA_PRED_PATH)
    score_col = "score_yes" if "score_yes" in qlora_pred.columns else "score"
    pred_col = "pred"
    if "sample_id" in qlora_pred.columns and "sample_id" in test_groups.columns:
        qdf = qlora_pred.merge(test_groups[["sample_id", ITEM_COL, "target_train_count", "item_count_bucket", "item_popularity_quantile", "history_len_bucket"]], on="sample_id", how="left")
        qrows = []
        for group_col in group_cols:
            for value, part in qdf.groupby(group_col, dropna=False):
                qrows.append({"method": "qwen_qlora_checkpoint2500", "group_type": group_col, "group": str(value), **calc_group_metrics(part["y_true"], part[score_col], part[pred_col])})
        qlora_group_metrics = pd.DataFrame(qrows)
        qlora_group_metrics.to_csv(OUT_DIR / "qlora_group_metrics.csv", index=False)
        q_sample_ids = set(qlora_pred["sample_id"].dropna().tolist())
        base_q = base_test_pred[base_test_pred["sample_id"].isin(q_sample_ids)]
        sem_q = sem_test_pred[sem_test_pred["sample_id"].isin(q_sample_ids)]
        fair_rows = [
            {"method": "catboost_popularity_on_qlora_subset", **calc_group_metrics(base_q["y_true"], base_q["score"], base_q["pred"])},
            {"method": "catboost_popularity_semantic_on_qlora_subset", **calc_group_metrics(sem_q["y_true"], sem_q["score"], sem_q["pred"])},
            {"method": "qwen_qlora_checkpoint2500", **calc_group_metrics(qlora_pred["y_true"], qlora_pred[score_col], qlora_pred[pred_col])},
        ]
        fair_subset_metrics = pd.DataFrame(fair_rows)
        fair_subset_metrics.to_csv(OUT_DIR / "fair_subset_metrics_on_qlora_samples.csv", index=False)
        display(fair_subset_metrics)
    else:
        print("QLoRA predictions do not contain sample_id; group analysis is skipped.")
else:
    print("QLoRA predictions were not found; QLoRA group analysis is skipped.")


## 9. Save summary report


In [ ]:
all_group_metrics = pd.concat([catboost_group_metrics, qlora_group_metrics], ignore_index=True)
all_group_metrics.to_csv(OUT_DIR / "all_group_metrics.csv", index=False)

summary = {
    "task": "long_tail_cold_start_analysis",
    "overall_metrics": overall_metrics.to_dict(orient="records"),
    "group_columns": group_cols,
    "feature_sets": {
        "base_features": FEATURES_BASE,
        "semantic_features": semantic_features,
        "features_with_semantic": FEATURES_SEM,
    },
    "outputs": {
        "overall_metrics": str(OUT_DIR / "overall_metrics.csv"),
        "catboost_group_metrics": str(OUT_DIR / "catboost_group_metrics.csv"),
        "qlora_group_metrics": str(OUT_DIR / "qlora_group_metrics.csv") if not qlora_group_metrics.empty else None,
        "fair_subset_metrics": str(OUT_DIR / "fair_subset_metrics_on_qlora_samples.csv") if not fair_subset_metrics.empty else None,
        "all_group_metrics": str(OUT_DIR / "all_group_metrics.csv"),
    }
}

with open(OUT_DIR / "long_tail_cold_start_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

report_path = PROJECT_ROOT / "reports" / "experiment_reports" / "long_tail_cold_start_analysis.md"
report_path.parent.mkdir(parents=True, exist_ok=True)

lines = []
lines.append("# Long-tail / cold-start analysis")
lines.append("")
lines.append("This report evaluates CatBoost, CatBoost with semantic features, and QLoRA across item popularity and user history groups.")
lines.append("")
lines.append("## Overall metrics")
lines.append("")
lines.append(overall_metrics.to_string(index=False))
lines.append("")
lines.append("## Fair subset metrics on QLoRA samples")
lines.append("")

if not fair_subset_metrics.empty:
    lines.append(fair_subset_metrics.to_string(index=False))
else:
    lines.append("QLoRA fair subset metrics were not available.")

lines.append("")
lines.append("## Group metrics")
lines.append("")
lines.append(all_group_metrics.to_string(index=False))
lines.append("")

report_text = "\n".join(lines)
report_path.write_text(report_text, encoding="utf-8")

print("saved output directory:", OUT_DIR)
print("saved report:", report_path)